# Chapter 3: Multiple Regression Analysis — Estimation

Topics: Multiple OLS in practice, OLS in matrix form, omitted variable bias, ceteris paribus interpretation.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import wooldridge as woo
import matplotlib.pyplot as plt
import os

output_dir = "output"
os.makedirs(output_dir, exist_ok=True)

## 3.1 Multiple Regression in Practice

In [ ]:
df = woo.dataWoo('wage1')

# Simple vs multiple regression
m1 = smf.ols('wage ~ educ', data=df).fit()
m2 = smf.ols('wage ~ educ + exper', data=df).fit()
m3 = smf.ols('wage ~ educ + exper + tenure', data=df).fit()

print("Simple regression (educ only):")
print(f"  β_educ = {m1.params['educ']:.4f}")
print("\nMultiple regression (educ + exper):")
print(f"  β_educ = {m2.params['educ']:.4f}")
print("\nFull model (educ + exper + tenure):")
print(m3.summary())

## 3.2 OLS in Matrix Form

In [ ]:
# Manual OLS via normal equations: β = (X'X)^{-1} X'y
X = np.column_stack([np.ones(len(df)), df['educ'], df['exper'], df['tenure']])
y = df['wage'].values

XtX_inv = np.linalg.inv(X.T @ X)
beta_hat = XtX_inv @ X.T @ y
print("OLS coefficients (matrix form):")
for name, val in zip(['Intercept','educ','exper','tenure'], beta_hat):
    print(f"  {name}: {val:.4f}")

# Verify against statsmodels
print("\nStatsmodels coefficients:")
print(m3.params.round(4))

## 3.3 Omitted Variable Bias

In [ ]:
# Omitting 'exper' from wage regression
m_full    = smf.ols('wage ~ educ + exper + tenure', data=df).fit()
m_omitted = smf.ols('wage ~ educ + tenure', data=df).fit()

print("With exper:   β_educ =", round(m_full.params['educ'], 4))
print("Without exper: β_educ =", round(m_omitted.params['educ'], 4))

# Correlation between omitted var and included vars
print(f"\nCorr(educ, exper) = {df['educ'].corr(df['exper']):.4f}")
print("Omitted var bias direction: negative (exper negatively correlated with educ, positively with wage)")

## Comparing Models: R² across specifications

In [ ]:
models = {
    'educ only': m1,
    'educ + exper': m2,
    'educ + exper + tenure': m3
}

fig, ax = plt.subplots(figsize=(8, 4))
names = list(models.keys())
r2    = [m.rsquared for m in models.values()]
ax.bar(names, r2, color=['#1f77b4','#ff7f0e','#2ca02c'])
ax.set_ylabel('R²')
ax.set_title('R² Across Model Specifications')
ax.set_ylim(0, max(r2)*1.3)
for i, v in enumerate(r2):
    ax.text(i, v + 0.002, f'{v:.3f}', ha='center')
plt.tight_layout()
plt.savefig(f"{output_dir}/fig1_r2_comparison.png", dpi=150)
plt.show()